# Capstone — mirrors your deployed research paper

**Ranking existing content for refresh review: a client-grouped validation audit.**

This notebook recomputes every number the paper claims, from the raw anonymized
starter dataset, out-of-fold, with a client-grouped split. It runs top to bottom
with a clean kernel — no hidden state, no cached figures.

> Research paper: `paper/index.html` (this repo). Every figure below is regenerated here.
> The validated model is the logistic regression under a client-grouped **GroupKFold k=5**;
> the transparent Week-4 rule is the baseline scored on the *same* folds.


## 1. Question

**Which content should a refresh hour spend its time on this week?**

- Unit of analysis: one pseudonymous content item (page) in the starter dataset.
- Output: a **ranked order** of the 30,000 items by model-estimated probability that
  the item is in the observed "declining" 30-day trend. A content editor opens the
  queue at rank 1, reads the reason code, and acts only after a human check.
- Wrong-call cost: an hour of editorial effort spent on a page that is not the best
  next candidate, while a better one waits.
- Why ML/statistics helps: a human cannot re-derive an ordering over 30,000 pages by
  hand, but can verify the first 20-50 recommendations quickly.


In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path

DATA = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA)
print("rows x cols:", df.shape)
print("distinct clients:", df['client_id'].nunique())

rows x cols: (30000, 44)
distinct clients: 32



## 2. Data

* **Source:** the in-repo anonymized starter dataset
  `data/raw/content_refresh_anonymized.csv` (30 000 rows x 44 columns) — one row per
  pseudonymous content item, 32 clients, trailing-90-day window metrics plus a
  30-day vs previous-30-day trend.
* **Excluded on purpose:** `client_id` (grouping key only, never a feature),
  `content_id` (row id), and anything that could identify content. The label-derived
  columns `trend_direction` and `trend_pct` are excluded from the model and used
  **only** to define the observed label.
* **Date windows:** all window columns are snapshot-time aggregates — no overlapping
  future windows.


In [2]:
# NOTE: no raw identifiers are printed here. content_id / client_id are
# pseudonymous and are used only for grouping / row identity — never features.
print("trend_direction value counts:")
print(df["trend_direction"].value_counts().to_string())
print()
df["decline_label"] = (df["trend_direction"] == "down").astype(int)
label = df["decline_label"]
print("declining items:", int(label.sum()), "| rate: %.3f" % label.mean())
print("non-declining :", int((1 - label).sum()))
print("rows:", len(df))

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

declining items: 16262 | rate: 0.542
non-declining : 13738
rows: 30000



## 3. Methodology

* **Assumption.** The observed label `decline = (trend_direction == 'down')` (30 vs
  previous 30 days, at snapshot time) is a workable proxy for "worth a human's
  refresh review". It is a proxy, not a crown truth.
* **Label:** `decline_label = (trend_direction == 'down')` — 16 262 / 30 000 rows (0.542).
* **Features (24):** 22 numeric + 2 categorical (`content_type`, `main_intent`).
  `trend_direction` / `trend_pct` are **never** features.
* **Baseline:** the transparent Week-4 rule — `stale (age >= 180d) + visible
  (impressions_90d >= 500) + ranking (avg_position >= 10)` → an integer 0..3 score
  ranked top-down. Fair comparison: it uses raw, human-checkable columns and the
  same rows, evaluated with the same metric on the same folds.
* **Validation:** client-grouped `GroupKFold(k=5)`; preprocessing refit inside each
  fold; every row scored out-of-fold by the fold that held its client out. The
  baseline rule is scored on the **same folds** — apples to apples.
* **Leakage checks:** (a) label sources excluded from features; (b) `client_id` is
  a grouping key only; (c) a probe that adds the label source as a feature reaches
  near-perfect AUC under the same harness — the model's 0.63 is nowhere near that,
  so the harness can detect leaks and finds none; (d) no future/overlapping windows.


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

CATEGORICAL_FEATURES = ["content_type", "main_intent"]
NUMERIC_FEATURES = [
    "search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions","content_age_days",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
forbidden = {"content_id","client_id","trend_direction","trend_pct","decline_label"}
assert set(FEATURES).isdisjoint(forbidden)
assert len(FEATURES) == 24 and len(set(FEATURES)) == 24

def build_pipeline():
    num = Pipeline([("imp", SimpleImputer(strategy="median")), ("scl", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([("num", num, NUMERIC_FEATURES), ("cat", cat, CATEGORICAL_FEATURES)])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=42))])

def pct_at_k(yt, pr, k):
    if len(yt) < k: return np.nan
    return float(np.mean(yt[np.argsort(pr)[::-1][:k]]))

# W4 baseline rule (raw columns, no training)
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["position_opportunity"] = (df["avg_position"] >= 10).astype(int)
df["baseline_score"] = df["stale"] + df["visible"] + df["position_opportunity"]

X = df[FEATURES]
y = df["decline_label"].to_numpy()
G = df["client_id"].to_numpy()
print("features:", len(FEATURES), "| label mean: %.3f" % y.mean())

features: 24 | label mean: 0.542



## 4. Results (model vs baseline, same splits)

One client-grouped `GroupKFold(5)` — identical folds for both. Mean ± std over the
five test folds. Base rate per fold is shown so every precision number is read
against the task's prevalence.

* **Model:** logistic regression (24 features, refit per fold), out-of-fold scores.
* **Baseline:** Week-4 0..3 rule scored on the identical folds.

Numbers are computed in the cell below from the raw `.csv` — nothing is hard-coded.


In [4]:
oof = np.full(len(df), np.nan)
base_score = df["baseline_score"].to_numpy()
rows = []
for i,(tr,va) in enumerate(GroupKFold(n_splits=5).split(X, y, groups=G)):
    assert set(G[tr]).isdisjoint(G[va]), "client leaked across folds"
    m = build_pipeline().fit(X.iloc[tr], y[tr])
    pv = m.predict_proba(X.iloc[va])[:, 1]
    oof[va] = pv
    yv = y[va]
    rows.append({
        "fold": i+1, "n": int(len(yv)), "base": round(float(yv.mean()), 3),
        "model_AUC":  round(float(roc_auc_score(yv, pv)), 3),
        "base_AUC":  round(float(roc_auc_score(yv, base_score[va])), 3),
        "model_P20": pct_at_k(yv, pv, 20),
        "base_P20": pct_at_k(yv, base_score[va], 20),
        "model_P50": pct_at_k(yv, pv, 50),
        "base_P50": pct_at_k(yv, base_score[va], 50),
        "model_P100": pct_at_k(yv, pv, 100),
        "base_P100": pct_at_k(yv, base_score[va], 100),
    })
folds = pd.DataFrame(rows)
print(folds.to_string(index=False))
print()
for col in ["model_AUC","base_AUC","model_P20","base_P20",
            "model_P50","base_P50","model_P100","base_P100"]:
    print("%s  mean %.3f  sd %.3f" % (col, folds[col].mean(), folds[col].std()))
print()
print("pooled base rate: %.3f" % float(y.mean()))
assert np.isfinite(oof).all(), "every row needs an out-of-fold score"
print("every row has an out-of-fold score: PASS")

 fold    n  base  model_AUC  base_AUC  model_P20  base_P20  model_P50  base_P50  model_P100  base_P100
    1 7008 0.490      0.606     0.435       0.75      0.35       0.78      0.36        0.74       0.45
    2 5731 0.645      0.580     0.486       0.30      0.65       0.48      0.58        0.53       0.58
    3 5753 0.379      0.659     0.609       0.80      0.40       0.72      0.42        0.77       0.52
    4 5755 0.622      0.662     0.671       0.75      0.90       0.68      0.88        0.76       0.84
    5 5753 0.585      0.652     0.587       0.80      0.90       0.88      0.78        0.83       0.74

model_AUC  mean 0.632  sd 0.037
base_AUC  mean 0.558  sd 0.096
model_P20  mean 0.680  sd 0.214
base_P20  mean 0.640  sd 0.263
model_P50  mean 0.708  sd 0.148
base_P50  mean 0.604  sd 0.224
model_P100  mean 0.726  sd 0.115
base_P100  mean 0.626  sd 0.161

pooled base rate: 0.542
every row has an out-of-fold score: PASS



## 5. Limitations (what this cannot claim)

* **No causal claim.** There is no refresh-event column and no before/after design
  in this repo. "Declining" is an association with a single snapshot's trend label.
* **One snapshot.** A single time cut at trailing windows; no seasonality; no
  outcome-feedback loop. Numbers may not transfer to another month or client mix.
* **Wide fold spread.** Model P@50 is 0.708 ± 0.148 — one client's fold can look
  much better or worse than another's. The queue is an ordering aid, not a promise.
* **Missing values.** `search_volume`/`competition`/`cpc` are missing on a large
  share of rows; the imputer fills the model, the baseline uses raw columns.
* **Priority labels are heuristics.** `high >= 0.65` / `medium >= 0.50` are
  *proposed* thresholds cut from the model score, not validated or measured.
* **No production claim.** No serving, drift monitoring, or retraining in this
  repo — it is an analysis artifact for the paper.


In [5]:
# Missingness measured (the same numbers the paper reports)
miss_cols = ["search_volume","competition","cpc","word_count","char_count","main_intent"]
miss = pd.DataFrame({
    "column": miss_cols,
    "missing_rows": [int(df[c].isna().sum()) for c in miss_cols],
})
miss["share"] = (miss["missing_rows"] / len(df)).round(3)
print(miss.to_string(index=False))

       column  missing_rows  share
search_volume          2468  0.082
  competition          2468  0.082
          cpc          2468  0.082
   word_count          7699  0.257
   char_count          7699  0.257
  main_intent          2374  0.079



## 6. Ranked recommendations (the action playbook)

The playbook (W7, full detail in the paper's "Ranked recommendations" section)
maps every row to **one archetype -> one action** with a machine-readable reason.
`priority` is a **proposed** 3-level heuristic cut from the model score, not
validated:

| priority band | model P(decline) | meaning |
|---|---|---|
| high | >= 0.65 | act first, after human review |
| medium | >= 0.50 | strong watch list |
| low | < 0.50 | monitor |

**Archetype -> action** (W7): `refresh_candidate -> refresh`;
`thin_content_candidate -> expand_and_refresh`; `ctr_candidate ->
refresh_and_review_ctr`; `engagement_candidate -> refresh_and_review_engagement`;
`ranking_candidate -> recheck_position`; `monitor_candidate -> monitor`;
`insufficient_data -> human_review`; `healthy -> monitor`.

**Intended use.** A content editor opens the queue at rank 1 and reviews rows in
order. Human sign-off is required (checks: intent fit, content quality, SERP
context, single-number trend check, business relevance, data completeness). The
reviewer may always reject. **No auto-refresh / no auto-publish.**

**No-go cases.** Any row carrying `no_position_data` / `no_keyword_data` /
`no_wordcount` -> `insufficient_data` — a reason to **not** act. `healthy` is
monitored, not touched.

**Monitoring / retrain triggers (proposed governance, not claims of having fired):**
AUC mean < ~0.55 on two consecutive re-validation runs; score-distribution drift
> ~2 sd; a sharp jump in no-evidence rows; reviewer override > ~20% on the top of
the queue; top-500 rank instability; label-definition change; fold-composition change.


In [6]:
# The playbook's priority mix on out-of-fold scores (W7's observed table)
df["model_probability"] = oof          # out-of-fold, honest
prio = pd.cut(df["model_probability"], bins=[-np.inf, 0.50, 0.65, np.inf],
              labels=["low","medium","high"])
print(prio.value_counts().sort_index().to_string())
n_risk = int((df["model_probability"] >= 0.50).sum())
print("rows with model P(decline) >= 0.50 (proposed threshold):", n_risk)

model_probability
low       10728
medium    10905
high       8367
rows with model P(decline) >= 0.50 (proposed threshold): 19272


## 7. Artifacts the paper embeds

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

FIG = Path("../../work/figures")
FIG.mkdir(parents=True, exist_ok=True)

In [8]:
# --- Figure 1: headline model vs baseline on the same folds ---
fig, ax = plt.subplots(figsize=(7, 4.2))
x = np.arange(2); w = 0.34
ax.bar(x - w/2, [folds["model_AUC"].mean(), folds["model_P50"].mean()], w,
       label="model (logistic)", color="#3572A5")
ax.bar(x + w/2, [folds["base_AUC"].mean(), folds["base_P50"].mean()], w,
       label="baseline rule", color="#D0A15A")
ax.axhline(float(y.mean()), color="grey", ls="--", lw=1, label="base rate %.3f" % y.mean())
ax.set_xticks(x); ax.set_xticklabels(["AUC", "Precision@50"])
ax.set_ylim(0.0, 1.0); ax.set_ylabel("metric value")
ax.set_title("Model vs Week-4 baseline — same client-grouped folds, out-of-fold")
ax.legend()
fig.savefig(FIG / "fig_model_vs_baseline.svg", bbox_inches="tight")
plt.close(fig)
print("saved", FIG / "fig_model_vs_baseline.svg")

saved ..\..\work\figures\fig_model_vs_baseline.svg


In [9]:
# --- Figure 2: precision at the head of the ranked queue vs base rate ---
order = np.argsort(df["model_probability"])[::-1]
yk = y[order]
head = pd.DataFrame({
    "k": ["top-20", "top-50", "top-100"],
    "precision": [round(float(yk[:20].mean()), 3),
                  round(float(yk[:50].mean()), 3),
                  round(float(yk[:100].mean()), 3)],
})
head["base"] = round(float(y.mean()), 3)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(head["k"], head["precision"], color="#3572A5")
ax.axhline(head["base"].iloc[0], color="grey", ls="--", lw=1, label="base rate")
ax.set_ylabel("precision"); ax.set_ylim(0, 1.1)
ax.legend(); ax.set_title("Queue-head precision (out-of-fold scores)")
fig.savefig(FIG / "fig_queue_head.svg", bbox_inches="tight"); plt.close(fig)
print(head.to_string(index=False))

      k  precision  base
 top-20       0.45 0.542
 top-50       0.64 0.542
top-100       0.71 0.542



**Self-check**

- [x] Every section above is filled — markdown thinking *and* the code that backs it.
- [x] This notebook runs top to bottom with no errors (executed in this file).
- [x] No client names, URLs, or private queries anywhere.
- [x] Claims use careful words: observed, measured, directional, decision-support.
- [x] Committed in the repo under `work/notebooks/` (commit log at restart).
